# Sources and Sequencing

## What you'll learn

- Start a pipeline by generating data from scratch (generative source)
- Start a pipeline by ingesting existing files (external ingest)
- Chain steps together so each consumes the previous step's output
- Read provenance graphs to trace data flow

**Prerequisites:** [Your First Pipeline](../01-getting-started/01-first-pipeline.ipynb)
and the source checkout from [Installation](../../getting-started/installation.md).
The ingest examples read CSV fixtures from that checkout.  
**Estimated time:** 10 minutes

---

Generate data or ingest existing files, then connect a sequence of
operations that transform and score it.

In [ ]:
from __future__ import annotations

from artisan.operations.curator import IngestData
from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager
from artisan.utils import find_project_root, tutorial_setup
from artisan.visualization import (
    build_macro_graph,
    build_micro_graph,
    inspect_metrics,
    inspect_pipeline,
)

In [ ]:
PROJECT_ROOT = find_project_root()
SOURCE_FILES = sorted((PROJECT_ROOT / "tests" / "fixtures" / "csv").glob("*.csv"))[:2]

### Reading the provenance graphs

Each example ends with a provenance graph. Here's how to read them:

- **Grey boxes** in a macro graph are steps; in a micro graph they are
  execution records, one per execution unit
- **Blue boxes** are artifacts produced by each step
- **Arrows** show data flow: which step consumed which artifacts, and which step produced them

---

## Pattern 1: Generative source

A generative source creates artifacts without inputs. This pipeline has one step.

In [ ]:
env = tutorial_setup("sources_generative")

pipeline = PipelineManager.create(
    name="sources_generative",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)

pipeline.run(operation=DataGenerator, name="generate", params={"count": 3, "seed": 42})
pipeline.finalize()  # wait for pending work and close the manager

In [ ]:
inspect_pipeline(env.delta_root)

`DataGenerator` produced 3 `data` artifacts from nothing. The `count`
parameter controls how many datasets are created.

In [ ]:
build_macro_graph(env.delta_root)

In [ ]:
build_micro_graph(env.delta_root)

The graph has one step, three outputs, and no input arrows.

---

## Pattern 2: External file ingest

When your data already exists on disk, `IngestData` brings it into the
provenance graph. Pass file paths as inputs — the framework promotes them
to `FileRefArtifact`s, then reads their content into `DataArtifact`s.

In [ ]:
env = tutorial_setup("sources_ingest")

pipeline = PipelineManager.create(
    name="sources_ingest",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)

pipeline.run(operation=IngestData, name="ingest", inputs=[str(f) for f in SOURCE_FILES])
pipeline.finalize()

In [ ]:
inspect_pipeline(env.delta_root)

Two input files produced two pairs of artifacts: a `data` artifact
(loaded CSV content) and a `file_ref` artifact (metadata reference to
the original path) for each file.

In [ ]:
build_macro_graph(env.delta_root)

In [ ]:
build_micro_graph(env.delta_root)

The ingest graph connects the input file references to the resulting data
artifacts, so you can trace each dataset to its original file.

---

## Pattern 3: Linear chain

A chain connects steps sequentially: each step consumes the previous
step's output. You wire them together with `pipeline.output("step_name", "role")`,
which creates an `OutputReference` — a lazy pointer that the framework
resolves at execution time.

For readability, the examples below assign `output = pipeline.output` so
wiring calls read as `output("step_name", "role")`.

This example builds a three-step chain:

```
DataGenerator  →  DataTransformer  →  MetricCalculator
 (generate)        (transform)          (metrics)
```

In [ ]:
env = tutorial_setup("sources_chain")

pipeline = PipelineManager.create(
    name="sources_chain",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

# Step 0: generate 2 datasets
pipeline.run(operation=DataGenerator, name="generate", params={"count": 2, "seed": 42})

# Step 1: scale each dataset by 0.5
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    params={"scale_factor": 0.5, "variants": 1, "seed": 100},
)

# Step 2: compute distribution metrics
pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
)

pipeline.finalize()

`inputs={"dataset": output("generate", "datasets")}` connects the `dataset`
input role to the generator's `datasets` output role. The framework resolves
the reference from Delta Lake at dispatch time.

In [ ]:
inspect_pipeline(env.delta_root)

Two datasets flowed through the entire chain. Each step consumed the
previous step's output and produced its own artifacts.

In [ ]:
inspect_metrics(env.delta_root, step_number=2)

`MetricCalculator` computed distribution statistics (min, max, median,
range) and summary statistics (coefficient of variation, row count) for
each transformed dataset.

In [ ]:
build_macro_graph(env.delta_root)

In [ ]:
build_micro_graph(env.delta_root)

Read the graph from left to right: step 0 generated datasets, step 1 transformed
them, and step 2 computed metrics from the transformed outputs.

---

## Combining patterns: ingest into a chain

Replace `DataGenerator` with `IngestData` to transform and score external files.

In [ ]:
env = tutorial_setup("sources_ingest_chain")

pipeline = PipelineManager.create(
    name="sources_ingest_chain",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

# Step 0: ingest CSV files from disk
pipeline.run(operation=IngestData, name="ingest", inputs=[str(f) for f in SOURCE_FILES])

# Step 1: transform the ingested data
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("ingest", "data")},
    params={"scale_factor": 2.0, "seed": 42},
)

# Step 2: compute metrics
pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
)

pipeline.finalize()

In [ ]:
inspect_pipeline(env.delta_root)

In [ ]:
build_macro_graph(env.delta_root)

In [ ]:
build_micro_graph(env.delta_root)

The downstream chain is unchanged. The first `pipeline.run()` call now uses
`IngestData`, whose output role is `"data"` instead of `"datasets"`.

## Next steps

- [Branching and Merging](02-branching-and-merging.ipynb) — fan-out to
  multiple branches, then recombine with Merge
- [Execution Flow](../../concepts/execution-flow.md) — how the framework
  resolves `OutputReference`s and dispatches work
- [Building a Pipeline](../../how-to-guides/building-a-pipeline.md) —
  task-oriented guide for common pipeline construction patterns